In [7]:
import numpy as np
import pandas as pd

In [8]:
def generate_trader_history(trader_id, persona="disciplined", num_trades=150):
    trades = []
    
    # Initial State
    account_balance = 50000.0  # ₹50,000 base capital
    loss_streak = 0
    peak_balance = account_balance
    prev_lot_size = 1.0  # Initialize base lot size
    drawdown = 0.0
    recent_pnl = 0.0 

    
    for t in range(num_trades):
        # 1. Establish behavior rules based on state & persona
        current_loss_streak = loss_streak
        current_drawdown = drawdown
        current_recent_pnl = recent_pnl
        if persona == "revenge_tilt" and current_loss_streak >= 2:
            # Tilt behavior triggered
            time_delta = np.random.exponential(scale=25) if np.random.rand() < 0.75 else np.random.exponential(scale=180)
            lot_mult = np.random.uniform(1.8, 4.0) if np.random.rand() < 0.70 else np.random.uniform(0.9, 1.5)
            is_unhedged = np.random.choice([0, 1], p=[0.20, 0.80])
        
        elif persona == "over_trader" and t > 60:
            # Fatigue behavior triggered halfway through session
            time_delta = np.random.exponential(scale=45) if np.random.rand() < 0.80 else np.random.exponential(scale=200)
            lot_mult = np.random.uniform(1.1, 2.2)
            is_unhedged = np.random.choice([0, 1], p=[0.40, 0.60])
            
        else:
            # Disciplined behavior (or normal state before tilt)
            time_delta = np.random.exponential(scale=450) if np.random.rand() < 0.90 else np.random.exponential(scale=40)
            lot_mult = np.random.uniform(0.8, 1.2) if np.random.rand() < 0.95 else np.random.uniform(1.5, 2.0)
            is_unhedged = np.random.choice([0, 1], p=[0.75, 0.25])
        
        # Track how much size changed relative to the previous trade
        position_size_change = round(lot_mult / prev_lot_size, 2)

        # 2. Behavior-Based Labeling Engine (Evaluates the actions objectively)
        is_impulsive_time = time_delta < 45
        is_aggressive_sizing = lot_mult > 1.8 or position_size_change > 1.7
        is_distressed_state = current_loss_streak >= 2 or current_drawdown > 10.0
        
        # Risk Rule: If they are in a distressed state AND execution breaks discipline
        if is_distressed_state and (is_aggressive_sizing or is_impulsive_time):
            base_label = 1
        else:
            base_label = 0
            
        # Retain a tiny 2% operational anomaly/noise factor
        label = 1 - base_label if np.random.rand() < 0.02 else base_label
        win_prob = 0.35 if label == 1 else 0.50

        # 2. Simulate Trade Outcome (Win/Loss)
        # Tilt trades have worse outcomes due to poor execution
        is_win = np.random.rand() < win_prob
        
        base_risk = 1000 * lot_mult
        if is_win:
            pnl = base_risk * np.random.uniform(1.0, 1.5)
            loss_streak = 0
        else:
            pnl = -base_risk * np.random.uniform(0.8, 1.2)
            loss_streak += 1
            
        account_balance += pnl

        # Prevent the simulated account from going below zero
        account_balance = max(account_balance, 0)

        peak_balance = max(peak_balance, account_balance)

        if peak_balance > 0:
            drawdown = ((peak_balance - account_balance) / peak_balance) * 100
        else:
            drawdown = 100.0
            
        prev_lot_size = lot_mult
        recent_pnl = pnl

        trades.append({
            "trader_id": trader_id,
            "persona": persona,
            "time_delta_seconds": round(time_delta, 1),
            "lot_size_multiplier": round(lot_mult, 2),
            "position_size_change": position_size_change,
            "is_unhedged": is_unhedged,
            "loss_streak_count": current_loss_streak,
            "recent_pnl_delta": round(current_recent_pnl, 2),
            "current_drawdown_pct": round(current_drawdown, 2),
            "label": label
        })
        
    return trades

In [20]:
# --- Build the Dataset ---
all_simulated_trades = []
trader_counter = 0

# Generate mixed profiles (30% Disciplined, 50% Revenge Tilters, 20% Over-traders)
configs = [("disciplined", 20), ("revenge_tilt", 35), ("over_trader", 15)]

for persona_type, count in configs:
    for _ in range(count):
        all_simulated_trades.extend(generate_trader_history(trader_counter, persona=persona_type))
        trader_counter += 1

df_advanced = pd.DataFrame(all_simulated_trades)


In [23]:
csv_filename = "ShieldFX_Synthetic_Trades_10k.csv"
df_advanced.to_csv(csv_filename, index=False)

In [24]:
print(f"Dataset Size: {df_advanced.shape}")

print("\nOverall Class Distribution:")
print(df_advanced["label"].value_counts())
print(df_advanced["label"].value_counts(normalize=True) * 100)

print("\nClass Distribution by Persona:")
print(pd.crosstab(df_advanced["persona"], df_advanced["label"]))

print("\nFeature Summary:")
print(df_advanced.describe())

Dataset Size: (10500, 10)

Overall Class Distribution:
label
0    7597
1    2903
Name: count, dtype: int64
label
0    72.352381
1    27.647619
Name: proportion, dtype: float64

Class Distribution by Persona:
label            0     1
persona                 
disciplined   2647   353
over_trader   1479   771
revenge_tilt  3471  1779

Feature Summary:
         trader_id  time_delta_seconds  lot_size_multiplier  \
count  10500.00000        10500.000000         10500.000000   
mean      34.50000          309.942695             1.316912   
std       20.20616          402.193852             0.645464   
min        0.00000            0.000000             0.800000   
25%       17.00000           37.500000             0.950000   
50%       34.50000          149.450000             1.090000   
75%       52.00000          433.275000             1.350000   
max       69.00000         4060.200000             3.990000   

       position_size_change  is_unhedged  loss_streak_count  recent_pnl_delta  \


In [12]:
print("\n=== Feature Ranges ===")
print(df_advanced[
    [
        "time_delta_seconds",
        "lot_size_multiplier",
        "position_size_change",
        "is_unhedged",
        "loss_streak_count",
        "recent_pnl_delta",
        "current_drawdown_pct"
    ]
].describe().T)

print("\n=== Label vs Features ===")
print(
    df_advanced.groupby("label")[
        [
            "time_delta_seconds",
            "lot_size_multiplier",
            "position_size_change",
            "is_unhedged",
            "loss_streak_count",
            "recent_pnl_delta",
            "current_drawdown_pct"
        ]
    ].mean()
)


=== Feature Ranges ===
                        count        mean          std      min       25%  \
time_delta_seconds    10500.0  312.329238   403.495922     0.00    36.100   
lot_size_multiplier   10500.0    1.329172     0.661194     0.80     0.940   
position_size_change  10500.0    1.100853     0.552078     0.22     0.840   
is_unhedged           10500.0    0.383333     0.486222     0.00     0.000   
loss_streak_count     10500.0    1.290571     1.862971     0.00     0.000   
recent_pnl_delta      10500.0   -9.325787  1657.333607 -4669.55 -1135.465   
current_drawdown_pct  10500.0   18.621593    21.918054     0.00     2.380   

                          50%        75%      max  
time_delta_seconds    150.000   438.4000  3232.20  
lot_size_multiplier     1.080     1.3800     4.00  
position_size_change    1.000     1.1900     4.90  
is_unhedged             0.000     1.0000     1.00  
loss_streak_count       1.000     2.0000    15.00  
recent_pnl_delta     -786.915  1292.3775  5888.